# INICIO

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import librosa
import librosa.display
import random
from IPython.display import Audio, display
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import soundfile as sf
from pathlib import Path
import yaml
from PIL import Image 
# Importar config
import sys
sys.path.append('../src')
from config import AUDIO_DIR, CSV_MONO, CSV_POLY, SAMPLE_RATE, N_FFT, HOP_LENGTH, OUTPUT_ROOT, SPECTROGRAM_DIR, LABELS_DIR

In [ ]:
# Para máxima resolución usamos los parámetros originales del dataset (44.1kHz)
SR_ANALISIS = 44100  # resolución nativa de DataSED
N_FFT_ANALISIS = 4096
HOP_ANALISIS = 512

In [ ]:
# Cargar CSV para saber qué audios tienen eventos
df = pd.read_csv(CSV_MONO)
audios_unicos = df['sound_name'].unique().tolist()
print(f"Total de audios con anotaciones: {len(audios_unicos)}")

# Funciones

In [ ]:
def analizar_duraciones(audio_dir):
    """
    Analiza las duraciones de todos los audios de una carpeta.
    Muestra estadísticas, el audio más corto y el más largo, y un histograma.
    
    Parámetros:
    -----------
    audio_dir : Path
        Carpeta donde están los archivos .wav
    """
    audios = sorted([f.name for f in Path(audio_dir).glob("*.wav")])
    print(f"Audios encontrados: {len(audios)}\n")

    duraciones = []
    for audio in audios:
        try:
            info = sf.info(Path(audio_dir) / audio)
            duraciones.append({
                'audio': audio,
                'duracion': info.duration,
                'sr': info.samplerate,
                'canales': info.channels
            })
        except Exception as e:
            print(f"Error leyendo {audio}: {e}")

    df_dur = pd.DataFrame(duraciones)

    # --- Estadísticas ---
    print("=" * 60)
    print("ESTADÍSTICAS DE DURACIÓN")
    print("=" * 60)
    print(df_dur['duracion'].describe())
    print(f"\nDuración mínima:     {df_dur['duracion'].min():.2f} s")
    print(f"Duración máxima:     {df_dur['duracion'].max():.2f} s")
    print(f"Desviación estándar: {df_dur['duracion'].std():.2f} s")
    print(f"¿Todas iguales?      {df_dur['duracion'].nunique() == 1}")

    # --- Audio más corto y más largo ---
    idx_min = df_dur['duracion'].idxmin()
    idx_max = df_dur['duracion'].idxmax()

    print("\n" + "=" * 60)
    print("AUDIO MÁS CORTO")
    print("=" * 60)
    print(f"Nombre:    {df_dur.loc[idx_min, 'audio']}")
    print(f"Duración:  {df_dur.loc[idx_min, 'duracion']:.2f} s")
    print(f"SR:        {df_dur.loc[idx_min, 'sr']} Hz")
    print(f"Canales:   {df_dur.loc[idx_min, 'canales']}")

    print("\n" + "=" * 60)
    print("AUDIO MÁS LARGO")
    print("=" * 60)
    print(f"Nombre:    {df_dur.loc[idx_max, 'audio']}")
    print(f"Duración:  {df_dur.loc[idx_max, 'duracion']:.2f} s "
          f"({df_dur.loc[idx_max, 'duracion']/60:.2f} min)")
    print(f"SR:        {df_dur.loc[idx_max, 'sr']} Hz")
    print(f"Canales:   {df_dur.loc[idx_max, 'canales']}")

    # --- Top 5 más cortos y más largos ---
    print("\n" + "=" * 60)
    print("TOP 5 AUDIOS MÁS CORTOS")
    print("=" * 60)
    print(df_dur.nsmallest(5, 'duracion')[['audio', 'duracion']].to_string(index=False))

    print("\n" + "=" * 60)
    print("TOP 5 AUDIOS MÁS LARGOS")
    print("=" * 60)
    print(df_dur.nlargest(5, 'duracion')[['audio', 'duracion']].to_string(index=False))

    # --- Histograma ---
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.hist(df_dur['duracion'], bins=40, color='#4ECDC4', edgecolor='black')
    ax.axvline(df_dur['duracion'].median(), color='yellow', linestyle='--',
               label=f"Mediana: {df_dur['duracion'].median():.1f} s")
    ax.axvline(df_dur['duracion'].mean(), color='salmon', linestyle='--',
               label=f"Media: {df_dur['duracion'].mean():.1f} s")
    ax.set_xlabel("Duración (s)")
    ax.set_ylabel("Número de audios")
    ax.set_title("Distribución de duraciones de los audios")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return df_dur

In [ ]:
df_anotaciones = pd.read_csv(CSV_MONO)

_PALETA = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#A78BFA', '#F472B6',
           '#34D399', '#FB923C', '#60A5FA', '#F87171', '#C084FC',
           '#2DD4BF', '#FBBF24', '#818CF8', '#FCA5A5', '#6EE7B7']

def analizar_audio(audio_name, guardar=False):
    audio_path = AUDIO_DIR / audio_name
    y, sr = librosa.load(audio_path, sr=SR_ANALISIS, mono=True)
    duracion = len(y) / sr

    # ============================================================
    # FFT DEL AUDIO COMPLETO
    # ============================================================
    n_fft_completo = int(2 ** np.ceil(np.log2(len(y))))
    fft_completa = np.fft.rfft(y, n=n_fft_completo)
    freqs_fft = np.fft.rfftfreq(n_fft_completo, 1/sr)
    magnitud_fft = np.abs(fft_completa)
    potencia_db_fft = 10 * np.log10(magnitud_fft**2 + 1e-10)
    potencia_db_fft -= potencia_db_fft.max()   # normaliza a su propio máximo

    # ============================================================
    # STFT
    # ============================================================
    stft_complejo = librosa.stft(y, n_fft=N_FFT_ANALISIS, hop_length=HOP_ANALISIS)
    magnitud_stft = np.abs(stft_complejo)
    freqs_stft = librosa.fft_frequencies(sr=sr, n_fft=N_FFT_ANALISIS)
    tiempos = librosa.frames_to_time(np.arange(magnitud_stft.shape[1]), sr=sr, hop_length=HOP_ANALISIS)
    magnitud_db_stft = librosa.amplitude_to_db(magnitud_stft, ref=np.max)

    potencia_stft_prom = np.mean(magnitud_stft**2, axis=1)
    potencia_db_stft = 10 * np.log10(potencia_stft_prom + 1e-10)
    potencia_db_stft -= potencia_db_stft.max()  # normaliza a su propio máximo

    # ============================================================
    # ANOTACIONES
    # ============================================================
    df_audio = df_anotaciones[df_anotaciones['sound_name'] == audio_name].copy()

    # ============================================================
    # ESCALAS
    # ============================================================
    FREQ_MIN, FREQ_MAX = 20, 20000
    DB_MIN, DB_MAX = -80, 0

    cgraf = "#969696"

    plt.style.use('dark_background')
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(
        f"Análisis de {audio_name} — {len(df_audio)} eventos "
        f"(SR={sr} Hz, dur={duracion:.1f}s)",
        fontsize=13, color='white'
    )

    # --- 1. Espectro de potencia FFT (eje X log 20 Hz–20 kHz) ---
    ax = axes[0, 0]
    mask_fft = (freqs_fft >= FREQ_MIN) & (freqs_fft <= FREQ_MAX)
    ax.semilogx(freqs_fft[mask_fft], potencia_db_fft[mask_fft],
                color=cgraf, linewidth=0.8)
    ax.set_xlim(FREQ_MIN, FREQ_MAX)
    ax.set_ylim(DB_MIN, DB_MAX)
    ax.set_xlabel("Frecuencia (Hz)")
    ax.set_ylabel("Potencia (dB)")
    ax.set_title("Espectro de Potencia FFT")
    ax.grid(True, alpha=0.2, which='both')
    ax.set_xticks([20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000])
    ax.set_xticklabels(['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k'])

    # --- 2. Espectro de potencia STFT (eje X log 20 Hz–20 kHz) ---
    ax = axes[0, 1]
    mask_stft = (freqs_stft >= FREQ_MIN) & (freqs_stft <= FREQ_MAX)
    ax.semilogx(freqs_stft[mask_stft], potencia_db_stft[mask_stft],
                color=cgraf, linewidth=0.8)
    ax.set_xlim(FREQ_MIN, FREQ_MAX)
    ax.set_ylim(DB_MIN, DB_MAX)
    ax.set_xlabel("Frecuencia (Hz)")
    ax.set_ylabel("Potencia (dB)")
    ax.set_title("Espectro de Potencia STFT (promediado)")
    ax.grid(True, alpha=0.2, which='both')
    ax.set_xticks([20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000])
    ax.set_xticklabels(['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k'])

    # --- 3. Espectrograma con etiquetas ---
    ax = axes[1, 0]
    img = ax.imshow(
        magnitud_db_stft, aspect='auto', origin='lower',
        extent=[tiempos[0], tiempos[-1], freqs_stft[0], freqs_stft[-1]],
        cmap='gray', vmin=DB_MIN, vmax=DB_MAX
    )
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Frecuencia (Hz)")
    ax.set_title("Espectrograma de Magnitud (con etiquetas)")

    clases_unicas = df_audio['class_name'].unique().tolist()
    color_por_clase = {c: _PALETA[i % len(_PALETA)] for i, c in enumerate(clases_unicas)}

    for _, row in df_audio.iterrows():
        t_ini, t_fin = row['start_time'], row['end_time']
        clase = row['class_name']
        color = color_por_clase[clase]
        rect = patches.Rectangle(
            (t_ini, freqs_stft[0]),
            t_fin - t_ini, freqs_stft[-1] - freqs_stft[0],
            linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.15
        )
        ax.add_patch(rect)
        ax.text(
            (t_ini + t_fin) / 2, freqs_stft[-1] * 0.95,
            clase, ha='center', va='top', fontsize=7, color='white',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85, edgecolor='none')
        )

    # --- 4. Espectrograma sin etiquetas ---
    ax = axes[1, 1]
    ax.imshow(
        magnitud_db_stft, aspect='auto', origin='lower',
        extent=[tiempos[0], tiempos[-1], freqs_stft[0], freqs_stft[-1]],
        cmap='gray', vmin=DB_MIN, vmax=DB_MAX
    )
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Frecuencia (Hz)")
    ax.set_title("Espectrograma de Magnitud (sin etiquetas)")

    # Colorbar compartida
    cbar = fig.colorbar(img, ax=axes[1, :], fraction=0.025, pad=0.02)
    cbar.set_label("Magnitud (dB)", color='white')
    cbar.ax.yaxis.set_tick_params(color='white')

    fig.subplots_adjust(left=0.06, right=0.92, top=0.93, bottom=0.07,
                        wspace=0.18, hspace=0.28)

    if guardar:
        out_path = Path("results/figures") / f"analisis_{audio_name.replace('.wav', '.png')}"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=100, bbox_inches='tight', facecolor='black')
        print(f"Guardado: {out_path}")

    plt.show()
    return fig

In [ ]:
def escuchar_audio(audio_name, audio_dir=AUDIO_DIR, sr=None):
    audio_path = Path(audio_dir) / audio_name
    if not audio_path.exists():
        print(f"❌ No existe: {audio_path}")
        return
    
    # Cargar con librosa (permite resamplear si sr != None)
    y, sr_real = librosa.load(audio_path, sr=sr, mono=True)
    duracion = len(y) / sr_real
    
    print(f"🎧 {audio_name}")
    print(f"   Duración: {duracion:.2f} s")
    print(f"   Sample rate: {sr_real} Hz")
    print(f"   Muestras: {len(y)}")
    
    display(Audio(data=y, rate=sr_real))

In [ ]:
def verificar_ventana(nombre_ventana=None):
    """
    Verifica una ventana del dataset YOLO generado.
    
    - Muestra en un mismo plot: imagen sola (izquierda) e imagen con cajas (derecha).
    - Imprime el contenido del .txt de la etiqueta y del data.yaml.
    
    Parametros:
    -----------
    nombre_ventana : str o None
        Nombre de la ventana (ej. "X-0000" o "X-0000.png").
        Si es None, se usa la primera imagen encontrada.
    """
    data_yaml_path = OUTPUT_ROOT / "data.yaml"
    
    # Cargar clases desde data.yaml
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names', {})
    if isinstance(names, dict):
        clases = [names[i] for i in sorted(names.keys())]
    else:
        clases = names
    
    # Determinar imagen a verificar
    imagenes = sorted(SPECTROGRAM_DIR.glob("*.png"))
    if not imagenes:
        print(f"No hay imagenes en {SPECTROGRAM_DIR}")
        return
    
    if nombre_ventana is not None:
        if not nombre_ventana.endswith('.png'):
            nombre_ventana += '.png'
        img_path = SPECTROGRAM_DIR / nombre_ventana
    else:
        img_path = imagenes[0]
    
    label_path = LABELS_DIR / (img_path.stem + ".txt")
    
    # Cargar imagen
    img = Image.open(img_path)
    ancho_img, alto_img = img.size
    
    print("=" * 60)
    print(f"IMAGEN: {img_path.name}")
    print("=" * 60)
    print(f"Modo:   {img.mode}")
    print(f"Tamano: {ancho_img} x {alto_img} px")
    
    # Leer etiquetas
    lineas = []
    if label_path.exists():
        with open(label_path, 'r') as f:
            lineas = [l.strip() for l in f if l.strip()]
    
    # --- Figura con dos subplots ---
    plt.style.use('dark_background')
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    fig.suptitle(
        f"{img_path.name}  —  Modo: {img.mode}  —  Tamano: {ancho_img} x {alto_img} px  —  Cajas: {len(lineas)}",
        fontsize=12, color='white'
    )
    
    # 1. Imagen sola (izquierda)
    ax = axes[0]
    ax.imshow(img, cmap='gray')
    ax.set_title("Imagen sin etiquetas")
    ax.axis('off')
    
    # 2. Imagen con cajas (derecha)
    ax = axes[1]
    ax.imshow(img, cmap='gray')
    ax.set_title("Imagen con etiquetas YOLO")
    
    if lineas:
        colores = plt.cm.tab20.colors
        for linea in lineas:
            partes = linea.split()
            clase_id = int(partes[0])
            x_c, y_c, w, h = map(float, partes[1:])
            
            x_c_px = x_c * ancho_img
            w_px = w * ancho_img
            y_c_px = y_c * alto_img
            h_px = h * alto_img
            x0 = x_c_px - w_px / 2
            y0 = y_c_px - h_px / 2
            
            color = colores[clase_id % len(colores)]
            rect = patches.Rectangle(
                (x0, y0), w_px, h_px,
                linewidth=2, edgecolor=color, facecolor=color, alpha=0.2
            )
            ax.add_patch(rect)
            
            nombre = clases[clase_id] if clase_id < len(clases) else f"id_{clase_id}"
            ax.text(
                x0 + 5, y0 + 15, nombre,
                fontsize=7, color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8, edgecolor='none')
            )
    
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    # --- Imprimir etiqueta ---
    print("=" * 60)
    print(f"ETIQUETA: {label_path.name}")
    print("=" * 60)
    if not lineas:
        print("(sin eventos anotados en esta ventana)")
    else:
        print(f"Total de cajas: {len(lineas)}")
        print(f"{'clase':<40} {'x_center':>10} {'y_center':>10} {'w':>10} {'h':>10}")
        print("-" * 85)
        for linea in lineas:
            partes = linea.split()
            clase_id = int(partes[0])
            x_c, y_c, w, h = map(float, partes[1:])
            nombre = clases[clase_id] if clase_id < len(clases) else f"id_{clase_id}"
            print(f"{nombre:<40} {x_c:>10.4f} {y_c:>10.4f} {w:>10.4f} {h:>10.4f}")

In [ ]:
def analizar_dataset_generado():
    """
    Analisis estadistico de las imagenes y etiquetas generadas.
    - Modos de imagen, tamanios, rango de valores.
    - Numero de cajas por imagen, distribucion de clases.
    - Imagenes sin cajas (ejemplos negativos).
    """
    imagenes = sorted(SPECTROGRAM_DIR.glob("*.png"))
    labels = sorted(LABELS_DIR.glob("*.txt"))

    print("=" * 60)
    print("ESTADISTICAS DEL DATASET GENERADO")
    print("=" * 60)

    # 1. Conteo basico
    print(f"Imagenes: {len(imagenes)}")
    print(f"Labels:   {len(labels)}")
    print(f"Coinciden: {len(imagenes) == len(labels)}")

    # 2. Modos y tamanios de imagen
    modos = Counter()
    tamanios = Counter()
    for p in imagenes:
        img = Image.open(p)
        modos[img.mode] += 1
        tamanios[img.size] += 1
    print(f"\nModos de imagen: {dict(modos)}")
    print(f"Tamanios unicos: {dict(tamanios)}")

    # 3. Cajas por imagen y labels vacios
    n_cajas_por_img = []
    labels_vacios = 0
    todas_clases = Counter()
    clase_id_a_nombre = {}
    # Leer nombres de clases desde data.yaml
    import yaml
    with open(OUTPUT_ROOT / "data.yaml", 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names', {})
    if isinstance(names, dict):
        clase_id_a_nombre = {i: names[i] for i in sorted(names.keys())}
    else:
        clase_id_a_nombre = {i: n for i, n in enumerate(names)}

    for p in labels:
        with open(p, 'r') as f:
            lineas = [l.strip() for l in f if l.strip()]
        n_cajas_por_img.append(len(lineas))
        if len(lineas) == 0:
            labels_vacios += 1
        for linea in lineas:
            clase_id = int(linea.split()[0])
            todas_clases[clase_id] += 1

    n_cajas_por_img = np.array(n_cajas_por_img)
    print(f"\nCajas por imagen:")
    print(f"  Media:   {n_cajas_por_img.mean():.2f}")
    print(f"  Mediana: {np.median(n_cajas_por_img):.1f}")
    print(f"  Min:     {n_cajas_por_img.min()}")
    print(f"  Max:     {n_cajas_por_img.max()}")
    print(f"  Labels vacios: {labels_vacios} ({100*labels_vacios/len(labels):.1f}%)")

    # 4. Distribucion de clases
    print(f"\nDistribucion de clases (total de cajas: {sum(todas_clases.values())}):")
    for clase_id, count in sorted(todas_clases.items(), key=lambda x: -x[1]):
        nombre = clase_id_a_nombre.get(clase_id, f"id_{clase_id}")
        print(f"  {nombre:<45} {count:>5}")

    # 5. Histograma de cajas por imagen
    plt.style.use('dark_background')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.hist(n_cajas_por_img, bins=range(0, n_cajas_por_img.max() + 2),
            color='#969696', edgecolor='black', align='left')
    ax.set_xlabel("Numero de cajas por imagen")
    ax.set_ylabel("Numero de imagenes")
    ax.set_title("Distribucion de cajas por imagen")
    ax.grid(True, alpha=0.2)

    ax = axes[1]
    clases_ordenadas = sorted(todas_clases.items(), key=lambda x: -x[1])
    nombres = [clase_id_a_nombre.get(c, str(c)) for c, _ in clases_ordenadas]
    valores = [v for _, v in clases_ordenadas]
    ax.barh(range(len(nombres)), valores, color='#969696')
    ax.set_yticks(range(len(nombres)))
    ax.set_yticklabels(nombres, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Numero de cajas")
    ax.set_title("Distribucion de clases")
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    # 6. Rango de valores de una imagen de ejemplo (16 bits)
    img_ejemplo = Image.open(imagenes[0])
    extrema = img_ejemplo.getextrema()
    print(f"\nRango de valores de {imagenes[0].name}: {extrema}")

    return {
        'n_imagenes': len(imagenes),
        'n_labels': len(labels),
        'modos': dict(modos),
        'tamanios': dict(tamanios),
        'n_cajas_por_img': n_cajas_por_img,
        'labels_vacios': labels_vacios,
        'clases': dict(todas_clases),
    }

In [ ]:
def verificar_split():
    """
    Verifica la distribucion train/val/test del dataset YOLO.

    Comprueba:
    - Existencia de las carpetas y conteo de imagenes/labels por split.
    - Coincidencia 1:1 entre imagenes y labels en cada split.
    - Que no haya imagenes duplicadas entre splits (data leakage).
    - Distribucion de clases por split y comparacion de proporciones.
    - Grafica comparativa de la distribucion de clases.
    """
    data_yaml_path = OUTPUT_ROOT / "data.yaml"

    # Cargar clases desde data.yaml
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names', {})
    if isinstance(names, dict):
        clase_id_a_nombre = {i: names[i] for i in sorted(names.keys())}
    else:
        clase_id_a_nombre = {i: n for i, n in enumerate(names)}

    print("=" * 60)
    print("VERIFICACION DEL SPLIT TRAIN / VAL / TEST")
    print("=" * 60)

    splits = ['train', 'val', 'test']
    resumen = {}

    # ------------------------------------------------------------
    # 1. Conteo de archivos por split
    # ------------------------------------------------------------
    print("\n1. Conteo de archivos por split:")
    for split in splits:
        img_dir = OUTPUT_ROOT / "images" / split
        lbl_dir = OUTPUT_ROOT / "labels" / split
        imgs = sorted(img_dir.glob("*.png")) if img_dir.exists() else []
        lbls = sorted(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
        resumen[split] = {
            'imgs': imgs,
            'lbls': lbls,
            'stems_img': {p.stem for p in imgs},
            'stems_lbl': {p.stem for p in lbls},
        }
        print(f"  {split:<6} imagenes: {len(imgs):>5}   labels: {len(lbls):>5}")

    # ------------------------------------------------------------
    # 2. Coincidencia 1:1 imagen <-> label
    # ------------------------------------------------------------
    print("\n2. Coincidencia imagen <-> label por split:")
    for split in splits:
        s = resumen[split]
        faltan_lbl = s['stems_img'] - s['stems_lbl']
        faltan_img = s['stems_lbl'] - s['stems_img']
        ok = len(faltan_lbl) == 0 and len(faltan_img) == 0
        print(f"  {split:<6} {'OK' if ok else 'FALLA'}"
              f"   sin label: {len(faltan_lbl)}   sin imagen: {len(faltan_img)}")
        if faltan_lbl:
            print(f"         imagenes sin label: {sorted(faltan_lbl)[:5]}...")
        if faltan_img:
            print(f"         labels sin imagen: {sorted(faltan_img)[:5]}...")

    # ------------------------------------------------------------
    # 3. Data leakage: mismos stems en varios splits
    # ------------------------------------------------------------
    print("\n3. Data leakage (mismos archivos en varios splits):")
    stems_train = resumen['train']['stems_img']
    stems_val = resumen['val']['stems_img']
    stems_test = resumen['test']['stems_img']
    dup_train_val = stems_train & stems_val
    dup_train_test = stems_train & stems_test
    dup_val_test = stems_val & stems_test
    total_dup = len(dup_train_val) + len(dup_train_test) + len(dup_val_test)
    print(f"  train & val:  {len(dup_train_val)}")
    print(f"  train & test: {len(dup_train_test)}")
    print(f"  val & test:   {len(dup_val_test)}")
    print(f"  Total duplicados: {total_dup} {'OK' if total_dup == 0 else 'FALLA'}")

    # ------------------------------------------------------------
    # 4. Distribucion de clases por split
    # ------------------------------------------------------------
    print("\n4. Distribucion de clases por split (conteo de cajas):")
    conteo_por_split = {}
    for split in splits:
        contador = Counter()
        for lbl in resumen[split]['lbls']:
            with open(lbl, 'r') as f:
                for linea in f:
                    linea = linea.strip()
                    if not linea:
                        continue
                    clase_id = int(linea.split()[0])
                    contador[clase_id] += 1
        conteo_por_split[split] = contador

    # Tabla comparativa
    todas_clases = sorted(set().union(*[set(c.keys()) for c in conteo_por_split.values()]))
    print(f"\n  {'clase':<45} {'train':>7} {'val':>7} {'test':>7}")
    print("  " + "-" * 70)
    for cid in todas_clases:
        nombre = clase_id_a_nombre.get(cid, f"id_{cid}")
        n_train = conteo_por_split['train'].get(cid, 0)
        n_val = conteo_por_split['val'].get(cid, 0)
        n_test = conteo_por_split['test'].get(cid, 0)
        print(f"  {nombre:<45} {n_train:>7} {n_val:>7} {n_test:>7}")

    # ------------------------------------------------------------
    # 5. Comparacion de proporciones (chi-cuadrado aproximado)
    # ------------------------------------------------------------
    print("\n5. Comparacion de proporciones por clase (train vs val vs test):")
    total_train = sum(conteo_por_split['train'].values()) or 1
    total_val = sum(conteo_por_split['val'].values()) or 1
    total_test = sum(conteo_por_split['test'].values()) or 1

    print(f"\n  {'clase':<45} {'train%':>8} {'val%':>8} {'test%':>8} {'dif max':>8}")
    print("  " + "-" * 82)
    for cid in todas_clases:
        nombre = clase_id_a_nombre.get(cid, f"id_{cid}")
        p_train = 100 * conteo_por_split['train'].get(cid, 0) / total_train
        p_val = 100 * conteo_por_split['val'].get(cid, 0) / total_val
        p_test = 100 * conteo_por_split['test'].get(cid, 0) / total_test
        dif_max = max(abs(p_train - p_val), abs(p_train - p_test), abs(p_val - p_test))
        print(f"  {nombre:<45} {p_train:>7.2f}% {p_val:>7.2f}% {p_test:>7.2f}% {dif_max:>7.2f}%")

    # ------------------------------------------------------------
    # 6. Imagenes sin cajas (labels vacios) por split
    # ------------------------------------------------------------
    print("\n6. Imagenes sin cajas (labels vacios) por split:")
    for split in splits:
        vacios = 0
        for lbl in resumen[split]['lbls']:
            if lbl.stat().st_size == 0:
                vacios += 1
        total = len(resumen[split]['lbls']) or 1
        print(f"  {split:<6} {vacios:>5} / {total:<5} ({100*vacios/total:.1f}%)")

    # ------------------------------------------------------------
    # 7. Grafica comparativa
    # ------------------------------------------------------------
    plt.style.use('dark_background')
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # 7.1 Conteo absoluto por clase
    ctrain = "#3B3B3B"
    cval   = "#969696"
    ctest  = "#F0F0F0"
    ax = axes[0]
    x = np.arange(len(todas_clases))
    ancho = 0.25
    nombres = [clase_id_a_nombre.get(c, str(c)) for c in todas_clases]
    train_vals = [conteo_por_split['train'].get(c, 0) for c in todas_clases]
    val_vals = [conteo_por_split['val'].get(c, 0) for c in todas_clases]
    test_vals = [conteo_por_split['test'].get(c, 0) for c in todas_clases]
    ax.barh(x - ancho, train_vals, ancho, label='train', color=ctrain)
    ax.barh(x, val_vals, ancho, label='val', color=cval)
    ax.barh(x + ancho, test_vals, ancho, label='test', color=ctest)
    ax.set_yticks(x)
    ax.set_yticklabels(nombres, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Numero de cajas")
    ax.set_title("Conteo de cajas por clase y split")
    ax.legend()
    ax.grid(True, alpha=0.2)

    # 7.2 Proporcion relativa por clase
    ax = axes[1]
    train_p = [100 * v / total_train for v in train_vals]
    val_p = [100 * v / total_val for v in val_vals]
    test_p = [100 * v / total_test for v in test_vals]
    ax.barh(x - ancho, train_p, ancho, label='train', color=ctrain)
    ax.barh(x, val_p, ancho, label='val', color=cval)
    ax.barh(x + ancho, test_p, ancho, label='test', color=ctest)
    ax.set_yticks(x)
    ax.set_yticklabels(nombres, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Porcentaje de cajas (%)")
    ax.set_title("Proporcion relativa de clases por split")
    ax.legend()
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # 8. Resumen final
    # ------------------------------------------------------------
    print("\n" + "=" * 60)
    print("RESUMEN")
    print("=" * 60)
    print(f"  Train: {len(resumen['train']['imgs'])} imagenes")
    print(f"  Val:   {len(resumen['val']['imgs'])} imagenes")
    print(f"  Test:  {len(resumen['test']['imgs'])} imagenes")
    print(f"  Total: {sum(len(resumen[s]['imgs']) for s in splits)} imagenes")
    print(f"  Data leakage: {'OK (0 duplicados)' if total_dup == 0 else f'FALLA ({total_dup} duplicados)'}")

    return resumen, conteo_por_split

In [ ]:
# Parametros de la escala log
FREQ_MIN = 20.0
FREQ_MAX = 22000.0
N_FILAS_LOG = 1024
DB_MIN, DB_MAX = -80, 0

# Paleta para las clases
_PALETA = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#A78BFA', '#F472B6',
           '#34D399', '#FB923C', '#60A5FA', '#F87171', '#C084FC',
           '#2DD4BF', '#FBBF24', '#818CF8', '#FCA5A5', '#6EE7B7']


def espectrograma_log_audio(audio_name, sr=None, n_fft=2048, hop_length=1024,
                            n_filas_log=N_FILAS_LOG, cmap='gray',
                            guardar=False, out_path=None, mostrar_cajas=True):
    """
    Genera y muestra el espectrograma de un audio con eje de frecuencia
    logaritmico (20 Hz - 22 kHz).

    Si mostrar_cajas=True, genera un subplot con dos paneles:
    - Izquierda: espectrograma limpio.
    - Derecha: espectrograma con las cajas YOLO superpuestas (calculadas
      a partir del CSV mono, recortadas al rango de tiempo del audio).

    Parametros:
    -----------
    audio_name : str
        Nombre del archivo de audio (ej. "S-0001.wav").
    sr : int o None
        Sample rate. Si es None, usa el nativo del archivo.
    n_fft : int
        Tamano de la ventana FFT.
    hop_length : int
        Desplazamiento entre ventanas.
    n_filas_log : int
        Numero de filas de la imagen final (alto).
    cmap : str
        'gray' (fondo negro, eventos blancos) o 'gray_r' (invertido).
    guardar : bool
        Si True, guarda la imagen como PNG de 16 bits.
    out_path : Path o str o None
        Ruta de guardado. Si es None y guardar=True, usa el nombre del audio.
    mostrar_cajas : bool
        Si True, genera dos paneles (limpio + con cajas).

    Retorna:
    --------
    dict con 'magnitud_log' (matriz), 'freqs_log' (eje Y), 'tiempos' (eje X).
    """
    audio_path = AUDIO_DIR / audio_name
    y, sr_real = librosa.load(audio_path, sr=sr, mono=True)
    duracion = len(y) / sr_real

    # STFT lineal
    stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    magnitud = np.abs(stft)
    magnitud_db = librosa.amplitude_to_db(magnitud, ref=np.max)
    freqs_lineal = librosa.fft_frequencies(sr=sr_real, n_fft=n_fft)
    tiempos = librosa.frames_to_time(
        np.arange(magnitud_db.shape[1]), sr=sr_real, hop_length=hop_length
    )

    # Rejilla logaritmica
    freqs_log = np.logspace(np.log10(FREQ_MIN), np.log10(FREQ_MAX), n_filas_log)

    # Interpolar cada frame a la rejilla log
    n_frames = magnitud_db.shape[1]
    magnitud_log = np.zeros((n_filas_log, n_frames), dtype=np.float32)
    for i in range(n_frames):
        magnitud_log[:, i] = np.interp(
            freqs_log, freqs_lineal, magnitud_db[:, i]
        )

    # --- Etiquetas del audio (si existen) ---
    df_audio = df_anotaciones[df_anotaciones['sound_name'] == audio_name].copy()

    # --- Figura ---
    plt.style.use('dark_background')

    if mostrar_cajas:
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    else:
        fig, ax = plt.subplots(figsize=(14, 6))
        axes = [ax]

    def _dibujar_espectrograma(ax, con_cajas):
        img = ax.imshow(
            magnitud_log, aspect='auto', origin='lower',
            extent=[tiempos[0], tiempos[-1], freqs_log[0], freqs_log[-1]],
            cmap=cmap, vmin=DB_MIN, vmax=DB_MAX
        )
        ax.set_yscale('log')
        ax.set_ylim(FREQ_MIN, FREQ_MAX)
        ax.set_xlim(tiempos[0], tiempos[-1])
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Frecuencia (Hz, log)")
        ax.set_yticks([20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000])
        ax.set_yticklabels(['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k'])
        ax.grid(True, alpha=0.2, which='both')

        if con_cajas and len(df_audio) > 0:
            clases_unicas = df_audio['class_name'].unique().tolist()
            color_por_clase = {c: _PALETA[i % len(_PALETA)]
                               for i, c in enumerate(clases_unicas)}
            for _, row in df_audio.iterrows():
                t_ini = row['start_time']
                t_fin = row['end_time']
                clase = row['class_name']
                color = color_por_clase[clase]
                rect = patches.Rectangle(
                    (t_ini, FREQ_MIN), t_fin - t_ini, FREQ_MAX - FREQ_MIN,
                    linewidth=2, edgecolor=color, facecolor=color, alpha=0.15
                )
                ax.add_patch(rect)
                ax.text(
                    (t_ini + t_fin) / 2, FREQ_MAX * 0.7,
                    clase, ha='center', va='top', fontsize=7, color='white',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color,
                              alpha=0.85, edgecolor='none')
                )
        return img

    # Panel izquierdo (o unico): limpio
    img = _dibujar_espectrograma(axes[0], con_cajas=False)
    axes[0].set_title("Espectrograma limpio")

    # Panel derecho: con cajas
    if mostrar_cajas:
        _dibujar_espectrograma(axes[1], con_cajas=True)
        axes[1].set_title(f"Con cajas ({len(df_audio)} eventos)")

    fig.suptitle(
        f"{audio_name}  —  SR={sr_real} Hz, N_FFT={n_fft}, "
        f"Hop={hop_length}, Filas log={n_filas_log}",
        fontsize=12
    )

    cbar = fig.colorbar(img, ax=axes, fraction=0.025, pad=0.02)
    cbar.set_label("Magnitud (dB)", color='white')

    fig.subplots_adjust(left=0.06, right=0.92, top=0.90, bottom=0.10, wspace=0.18)
    plt.show()

    # --- Guardar como PNG 16 bits (opcional) ---
    if guardar:
        img_array = np.clip(
            (magnitud_log - DB_MIN) / (DB_MAX - DB_MIN), 0, 1
        )
        img_uint16 = (img_array * 65535).astype(np.uint16)
        if cmap == 'gray':
            img_uint16 = 65535 - img_uint16
        img_uint16 = np.flipud(img_uint16)
        img_pil = Image.fromarray(img_uint16)
        if img_pil.mode != 'I;16':
            img_pil = img_pil.convert('I;16')
        if out_path is None:
            out_path = Path(f"espectrograma_log_{audio_name.replace('.wav', '.png')}")
        img_pil.save(out_path, format='PNG')
        print(f"Guardado en: {out_path}")

    return {
        'magnitud_log': magnitud_log,
        'freqs_log': freqs_log,
        'tiempos': tiempos,
        'n_eventos': len(df_audio),
    }

# Analisis

In [ ]:
# Cargar ambos CSV
df_mono = pd.read_csv(CSV_MONO)
df_poly = pd.read_csv(CSV_POLY)

print("=" * 60)
print("MONOFÓNICO")
print("=" * 60)
print(f"Filas: {len(df_mono)}")
print(f"Columnas: {list(df_mono.columns)}")
print(f"Audios únicos: {df_mono['sound_name'].nunique()}")
print(f"Clases únicas: {df_mono['class_name'].nunique()}")
print()
print(df_mono.head(10))

print("\n" + "=" * 60)
print("POLIFÓNICO")
print("=" * 60)
print(f"Filas: {len(df_poly)}")
print(f"Audios únicos: {df_poly['sound_name'].nunique()}")
print(f"Clases únicas: {df_poly['class_name'].nunique()}")
print()
print(df_poly.head(10))

In [ ]:
df_duraciones = analizar_duraciones(AUDIO_DIR)

In [ ]:
# Evento de duración más larga
idx_max = df_mono['event_length'].idxmax()
evento_max = df_mono.loc[idx_max]

print("=" * 60)
print("EVENTO DE DURACIÓN MÁS LARGA")
print("=" * 60)
print(f"Audio:      {evento_max['sound_name']}")
print(f"Clase:      {evento_max['class_name']}")
print(f"Inicio:     {evento_max['start_time']:.2f} s")
print(f"Fin:        {evento_max['end_time']:.2f} s")
print(f"Duración:   {evento_max['event_length']:.2f} s "
      f"({evento_max['event_length']/60:.2f} min)")

# Top 10 eventos más largos
print("\n" + "=" * 60)
print("TOP 10 EVENTOS MÁS LARGOS")
print("=" * 60)
top10 = df_mono.nlargest(10, 'event_length')[
    ['sound_name', 'class_name', 'start_time', 'end_time', 'event_length']
]
print(top10.to_string(index=False))

# Estadísticas por clase de los eventos más largos
print("\n" + "=" * 60)
print("DURACIÓN MÁXIMA POR CLASE")
print("=" * 60)
max_por_clase = df_mono.groupby('class_name')['event_length'].max().sort_values(ascending=False)
print(max_por_clase)

# Graficas Audios RAW

In [ ]:
fig = analizar_audio("S-0001.wav")
escuchar_audio("S-0001.wav")

In [ ]:
fig = analizar_audio("S-0087.wav")
escuchar_audio("S-0087.wav")

In [ ]:
fig = analizar_audio("S-0011.wav")
escuchar_audio("S-0011.wav")

In [ ]:
fig = analizar_audio("S-0432.wav")
escuchar_audio("S-0432.wav")

In [ ]:
fig = analizar_audio("S-0399.wav")
escuchar_audio("S-0399.wav")

In [ ]:
cadena = "S-0629.wav"
fig = analizar_audio(cadena)
escuchar_audio(cadena)

In [ ]:
cadena = "S-0189.wav"
fig = analizar_audio(cadena)
escuchar_audio(cadena)

# Datos a ingresar en YOHO
Imagenes generadas con `dataset_builder.py`

In [ ]:
x = "1038"
verificar_ventana(f"X-{x}")

In [ ]:
x = "1772"
verificar_ventana(f"X-{x}")

In [ ]:
x = "2986"
verificar_ventana(f"X-{x}")

# Inspeccion DataSet Generado

In [ ]:
stats = analizar_dataset_generado()

In [ ]:
resumen, conteo = verificar_split()

## Espectrograma en escala Log

In [ ]:
m, f, t, n = espectrograma_log_audio("S-0399.wav")

In [ ]:
m, f, t, n = espectrograma_log_audio("S-0629.wav")

In [ ]:
m, f, t, n = espectrograma_log_audio("S-0001.wav")

In [ ]:
m, f, t, n = espectrograma_log_audio("S-0201.wav")

In [ ]:
m, f, t, n = espectrograma_log_audio("S-0009.wav")

In [ ]:
m, f, t, n = espectrograma_log_audio("S-0189.wav")